# ***Part 1***

## ***Imports***

In [1]:
import gzip
import json
import math
import os
import pickle
import subprocess
import sys
from itertools import product
from pathlib import Path
import warnings

## ***XGBoost***

In [ ]:
# Install dependencies from the repository root before running this notebook:
# %pip install -r requirements.txt

## ***Dataset***

In [3]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    for filename in filenames:
        print("   ", filename)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/n1sarga
/kaggle/input/datasets/n1sarga/book-recommender
    BX_Books.csv
    BX-Book-Ratings.csv
    BX-Users.csv


In [4]:
PROJECT_ROOT = Path.cwd()if not (PROJECT_ROOT / 'data' / 'raw').exists():    PROJECT_ROOT = PROJECT_ROOT.parentDATA_DIR = PROJECT_ROOT / 'data' / 'raw'assert DATA_DIR.exists(), f"Dataset folder not found: {DATA_DIR}"

In [5]:
BOOKS_FILE = DATA_DIR / 'BX_Books.csv'
USERS_FILE = DATA_DIR / 'BX-Users.csv'
RATINGS_FILE = DATA_DIR / 'BX-Book-Ratings.csv'

## ***Artifacts***

In [6]:
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
MODEL_DIR = ARTIFACTS_DIR / 'spark-als-model'

## ***Spark Session***

In [7]:
APP_NAME = 'SparkBookRecommenderNotebook'
MASTER = 'local[*]'
SHUFFLE_PARTITIONS = '4'
CHECKPOINT_DIR = PROJECT_ROOT / '.spark-checkpoint'

In [8]:
os.environ['SPARK_LOCAL_HOSTNAME'] = 'localhost'

## ***ALS HyperParameters***

In [9]:
ALS_RANK_VALUE = 20
ALS_MAX_ITER_VALUE = 10
ALS_REG_PARAM_VALUE = 0.1
ALS_SEED_VALUE = 42

## ***ALS Objective Function***
ALS (***Alternating Least Squares***) is the core collaborative filtering algorithm used to approximate the user-item interaction matrix.

## ***Matrix Factorization***

$$
R \approx UV^T
$$

Where,
1. **R** = Original user-item rating matrix  
2. **U** = User latent factor matrix  
3. **V** = Item latent factor matrix

## ***Optimization Objective***

$$
\|R - UV^T\|^2 + \lambda(\|U\|^2 + \|V\|^2)
$$

Where,

### First term: Minimizes prediction error between actual ratings and predicted ratings.

$$
\|R - UV^T\|^2
$$

### Regularization Term: Prevents overfitting by penalizing excessively large latent factor values.

$$
\lambda(\|U\|^2 + \|V\|^2)
$$

## ***HyperParameters***

## rank: Number of latent dimensions (hidden preference features).
## max_iter: Number of optimization cycles ALS performs.
## reg_param (λ): Regularization strength.
1. Higher λ → More generalization, less overfitting
2. Lower λ → Better fit to training data, higher overfitting risk

## ***Prediction Formula***

For user **u** and item **i**:

$$
\hat{r}_{ui} = U_u \cdot V_i
$$

## ***Prediction Values***
1. ALS_RANK_VALUE = 20
2. ALS_MAX_ITER_VALUE = 10
3. ALS_REG_PARAM_VALUE = 0.1
4. ALS_SEED_VALUE = 42

## ***Data Filtering***

In [10]:
MIN_USER_RATINGS = 200

# Explicit rating scale
MIN_RATING = 1.0
MAX_RATING = 10.0

## ***Benchmarking Rule***

In [11]:
MODEL_COMPARISON_TARGET_ROWS = 25000

## ***Train / Test / Val Split***

In [12]:
MODEL_COMPARISON_SPLIT = (0.7, 0.15, 0.15)

## ***Ranking Eval***

In [13]:
RANKING_TOP_K = 10

## ***Rating Threshold***

In [14]:
RELEVANT_RATING_THRESHOLD = 8.0

## ***Feature Engineering***

In [15]:
CURRENT_YEAR = 2026.0

In [16]:
MODEL_FEATURE_COLUMNS = [
    'user_avg_rating',          # User mean rating tendency
    'user_rating_count_log',    # User activity level
    'item_avg_rating',          # Book average quality
    'item_rating_count_log',    # Book popularity
    'author_avg_rating',        # Author popularity
    'author_rating_count_log',  # Author exposure
    'age_filled',               # User age
    'publication_year_filled',  # Book publication year
    'user_item_mean_gap',       # User taste vs item popularity gap
    'book_age',                 # Book recency
]

## ***HyperParameter Tuning***
$$
\text{Total ALS Combinations: } 2 \times 2 \times 2 = 8
$$

### ***ALS Grid***

In [17]:
ALS_TUNING_GRID = [
    {
        'rank': rank,
        'max_iter': max_iter,
        'reg_param': reg_param
    }
    for rank, max_iter, reg_param in product(
        [12, 24],
        [10, 15],
        [0.05, 0.1]
    )
]

### ***FM Grid***

In [18]:
FM_GRID = [
    {
        'factor_size': factor_size,
        'reg_param': reg_param,
        'step_size': step_size
    }
    for factor_size, reg_param, step_size in product(
        [8, 16],
        [0.0, 0.05],
        [0.05]
    )
]

### ***XGBRanker Grid***

In [19]:
XGB_RANKER_GRID = [
    {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'min_child_weight': min_child_weight
    }
    for n_estimators, max_depth, learning_rate, min_child_weight in product(
        [30, 60],
        [4, 6],
        [0.05],
        [1]
    )
]

## ***Sample***

In [20]:
SAMPLE_USER_ID = 11676
SAMPLE_ISBN = '0001944711'

# ***Part 2***

## ***PySpark ML Imports***
1. ***VectorAssembler***: Combines multiple numerical columns into one feature vector.
$$
X = [x_1, x_2, x_3, ..., x_n]
$$
2. ***StandardScaler***: Standardizes features so large-value columns do not dominate.
$$
z = \frac{x - \mu}{\sigma}
$$

Where:

* $x$ = original value
* $\mu$ = mean
* $\sigma$ = standard deviation

3. ***ALS***
$$
R \approx UV^T
$$
4. ***Window***: Used for row ranking and ordered calculations.
Assigning:
$$
item_id = row_number() - 1
$$

5. ***XGBRanker***
$$
NDCG@K
$$

In [21]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StandardScaler
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.recommendation import ALS
from pyspark.ml.regression import FMRegressor
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql import functions as F
from xgboost import XGBRanker

# ***Spark Session Helper***
$$
\text{SparkSession} \rightarrow \text{Load Data} \rightarrow \text{Preprocess} \rightarrow \text{Train Models}
$$

In [22]:
def get_notebook_spark_session(
    app_name=APP_NAME,
    master=MASTER,
    checkpoint_dir=CHECKPOINT_DIR
):
    os.environ.setdefault('SPARK_LOCAL_HOSTNAME', 'localhost')
    os.environ.setdefault('PYSPARK_PYTHON', sys.executable)
    os.environ.setdefault('PYSPARK_DRIVER_PYTHON', sys.executable)

    spark = (
        SparkSession.builder
        .appName(app_name)
        .master(master)
        .config('spark.sql.shuffle.partitions', SHUFFLE_PARTITIONS)
        .config('spark.driver.host', '127.0.0.1')
        .config('spark.driver.bindAddress', '127.0.0.1')
        .config('spark.pyspark.python', sys.executable)
        .config('spark.pyspark.driver.python', sys.executable)
        .getOrCreate()
    )

    if (
        os.name != 'nt'
        or os.environ.get('HADOOP_HOME')
        or os.environ.get('hadoop.home.dir')
    ):
        checkpoint_path = Path(checkpoint_dir)
        checkpoint_path.mkdir(parents=True, exist_ok=True)
        spark.sparkContext.setCheckpointDir(str(checkpoint_path))

    return spark

## ***Dataset Description***
- **BX-Books.csv** → Book metadata  
- **BX-Users.csv** → User demographic data  
- **BX-Book-Ratings.csv** → User-book ratings 

In [23]:
def read_book_crossing_csv(spark, file_path):
    return (
        spark.read.option('header', True)
        .option('delimiter', ';')
        .option('quote', '"')
        .option('escape', '"')
        .option('encoding', 'ISO-8859-1')
        .option('mode', 'DROPMALFORMED')
        .csv(str(file_path))
    )


def load_raw_datasets(spark, books_path=BOOKS_FILE, users_path=USERS_FILE, ratings_path=RATINGS_FILE):
    books_raw = read_book_crossing_csv(spark, books_path)
    users_raw = read_book_crossing_csv(spark, users_path)
    ratings_raw = read_book_crossing_csv(spark, ratings_path)
    return books_raw, users_raw, ratings_raw

## ***Spark Starting and Data Loading*** 

In [24]:
spark = get_notebook_spark_session()
books_raw, users_raw, ratings_raw = load_raw_datasets(spark)

print('Raw books rows:', books_raw.count())
print('Raw users rows:', users_raw.count())
print('Raw ratings rows:', ratings_raw.count())
print('Raw books columns:', books_raw.columns)
print('Raw users columns:', users_raw.columns)
print('Raw ratings columns:', ratings_raw.columns)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/02 07:48:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Raw books rows: 271379
Raw users rows: 278859
Raw ratings rows: 1149780
Raw books columns: ['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'Image-URL-S', 'Image-URL-M', 'Image-URL-L']
Raw users columns: ['User-ID', 'Location', 'Age']
Raw ratings columns: ['User-ID', 'ISBN', 'Book-Rating']


## ***Data Pre-Processing***
$$
\text{Raw Data}
\rightarrow
\text{Clean Data}
\rightarrow
\text{Indexed Numeric Data}
\rightarrow
\text{ALS-Compatible Matrix}
$$

In [25]:
def prepare_books(books_df):
    return (
        books_df.select(
            F.trim(F.col('ISBN')).alias('isbn'),
            F.col('Book-Title').alias('title'),
            F.col('Book-Author').alias('author'),
            F.col('Year-Of-Publication').alias('year_of_publication'),
            F.col('Publisher').alias('publisher'),
        )
        .dropna(subset=['isbn', 'title'])
        .filter(F.col('isbn') != '')
        .dropDuplicates(['isbn'])
    )

In [26]:
def prepare_users(users_df):
    return (
        users_df.select(
            F.col('User-ID').cast('int').alias('user_id'),
            F.col('Location').alias('location'),
            F.when(
                F.col('Age').isin('NULL', '', 'N/A'),
                None
            ).otherwise(F.col('Age')).cast('int').alias('age'),
        )
        .dropna(subset=['user_id'])
        .dropDuplicates(['user_id'])
    )

In [27]:
def prepare_ratings(ratings_df, min_user_ratings=0):
    cleaned = (
        ratings_df.select(
            F.col('User-ID').cast('int').alias('user_id'),
            F.trim(F.col('ISBN')).alias('isbn'),
            F.col('Book-Rating').cast('float').alias('rating'),
        )
        .dropna(subset=['user_id', 'isbn', 'rating'])
        .filter((F.col('isbn') != '') & (F.col('rating') > 0))
        .dropDuplicates(['user_id', 'isbn'])
    )

    if min_user_ratings > 0:
        active_users = (
            cleaned.groupBy('user_id')
            .count()
            .filter(F.col('count') > min_user_ratings)
            .select('user_id')
        )
        cleaned = cleaned.join(active_users, on='user_id', how='inner')

    return cleaned

In [28]:
def build_item_index(books_df, ratings_df):
    rated_books = ratings_df.select('isbn').distinct().join(books_df.select('isbn'), on='isbn', how='inner')
    window = Window.orderBy('isbn')
    isbn_mapping = rated_books.withColumn('item_id', (F.row_number().over(window) - 1).cast('int'))
    indexed_books = books_df.join(isbn_mapping, on='isbn', how='inner')
    indexed_ratings = ratings_df.join(isbn_mapping, on='isbn', how='inner')
    return indexed_books, indexed_ratings, isbn_mapping

In [29]:
books = prepare_books(books_raw).cache()
users = prepare_users(users_raw).cache()
ratings = prepare_ratings(ratings_raw, min_user_ratings=MIN_USER_RATINGS).cache()

In [30]:
indexed_books, indexed_ratings, isbn_mapping = build_item_index(books, ratings)
indexed_books = indexed_books.cache()
indexed_ratings = indexed_ratings.cache()
isbn_mapping = isbn_mapping.cache()

26/05/02 07:48:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 0

In [31]:
book_lookup = indexed_books.select('item_id', 'isbn', 'title', 'author').dropDuplicates(['item_id']).cache()

In [32]:
print('Clean books rows:', books.count())
print('Clean users rows:', users.count())
print('Clean ratings rows:', ratings.count())
print('Indexed books rows:', indexed_books.count())
print('Indexed ratings rows:', indexed_ratings.count())
print('Unique indexed items:', isbn_mapping.count())
warnings.filterwarnings("ignore")

Clean books rows: 271379


Clean users rows: 278857


Clean ratings rows: 69226


26/05/02 07:48:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 0

Indexed books rows: 47170


26/05/02 07:48:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 0

Indexed ratings rows: 62191


26/05/02 07:48:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 07:48:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/02 0

Unique indexed items: 47170


In [33]:
isbn_mapping.orderBy('item_id').show(5, truncate=False)
warnings.filterwarnings("ignore")

+----------+-------+
|isbn      |item_id|
+----------+-------+
|0000913154|0      |
|0001046438|1      |
|000104687X|2      |
|0001047213|3      |
|0001047973|4      |
+----------+-------+
only showing top 5 rows


## ***Diagnostic***
$$
\text{Raw Data}
\rightarrow
\text{Clean Data}
\rightarrow
\text{Explore Structure}
\rightarrow
\text{Choose Better Modeling Strategy}
$$

In [34]:
exploration_summary = {
    'raw_books_rows': books_raw.count(),
    'raw_users_rows': users_raw.count(),
    'raw_ratings_rows': ratings_raw.count(),
    'clean_books_rows': books.count(),
    'clean_users_rows': users.count(),
    'clean_ratings_rows': ratings.count(),
    'indexed_ratings_rows': indexed_ratings.count(),
    'unique_users_after_filter': ratings.select('user_id').distinct().count(),
    'unique_books_after_filter': ratings.select('isbn').distinct().count(),
    'indexed_items': isbn_mapping.count(),
}

In [35]:
print(json.dumps(exploration_summary, indent=2))

{
  "raw_books_rows": 271379,
  "raw_users_rows": 278859,
  "raw_ratings_rows": 1149780,
  "clean_books_rows": 271379,
  "clean_users_rows": 278857,
  "clean_ratings_rows": 69226,
  "indexed_ratings_rows": 62191,
  "unique_users_after_filter": 144,
  "unique_books_after_filter": 53817,
  "indexed_items": 47170
}


In [36]:
print('Rating distribution after cleaning:')
ratings.groupBy('rating').count().orderBy('rating').show(20, truncate=False)

Rating distribution after cleaning:
+------+-----+
|rating|count|
+------+-----+
|1.0   |295  |
|2.0   |301  |
|3.0   |478  |
|4.0   |802  |
|5.0   |9738 |
|6.0   |4298 |
|7.0   |10564|
|8.0   |15760|
|9.0   |12606|
|10.0  |14384|
+------+-----+



In [37]:
print('Most active users after cleaning:')
ratings.groupBy('user_id').count().orderBy(F.desc('count'), F.asc('user_id')).show(10, truncate=False)

Most active users after cleaning:
+-------+-----+
|user_id|count|
+-------+-----+
|11676  |8524 |
|98391  |5802 |
|153662 |1969 |
|189835 |1906 |
|23902  |1395 |
|76499  |1036 |
|171118 |1035 |
|235105 |1023 |
|16795  |968  |
|248718 |948  |
+-------+-----+
only showing top 10 rows


In [38]:
print('Most rated books after cleaning:')
(
    indexed_ratings.groupBy('item_id')
    .count()
    .join(book_lookup.select('item_id', 'isbn', 'title', 'author'), on='item_id', how='inner')
    .orderBy(F.desc('count'), F.asc('title'))
    .select('isbn', 'title', 'author', 'count')
    .show(10, truncate=False)
)

Most rated books after cleaning:
+----------+----------------------------------------------------------------+----------------+-----+
|isbn      |title                                                           |author          |count|
+----------+----------------------------------------------------------------+----------------+-----+
|0316666343|The Lovely Bones: A Novel                                       |Alice Sebold    |24   |
|043935806X|Harry Potter and the Order of the Phoenix (Book 5)              |J. K. Rowling   |20   |
|059035342X|Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))|J. K. Rowling   |19   |
|0439139597|Harry Potter and the Goblet of Fire (Book 4)                    |J. K. Rowling   |17   |
|0312195516|The Red Tent (Bestselling Backlist)                             |Anita Diamant   |16   |
|0345370775|Jurassic Park                                                   |Michael Crichton|15   |
|0440214041|The Pelican Brief                             

## ***ALS***
$$
R \approx UV^T
$$

### Where:
- **R** = User-item rating matrix  
- **U** = User latent factor matrix  
- **V** = Item latent factor matrix  

In [39]:
def train_als_model(ratings_df, rank=ALS_RANK_VALUE, max_iter=ALS_MAX_ITER_VALUE, reg_param=ALS_REG_PARAM_VALUE):
    als = ALS(
        userCol='user_id',
        itemCol='item_id',
        ratingCol='rating',
        rank=rank,
        maxIter=max_iter,
        regParam=reg_param,
        coldStartStrategy='nan',
        nonnegative=True,
        seed=ALS_SEED_VALUE,
    )
    return als.fit(ratings_df)

In [40]:
if not indexed_ratings.take(1):
    raise ValueError('Training data empty. Check dataset or filtering threshold.')

model = train_als_model(
    indexed_ratings,
    rank=ALS_RANK_VALUE,
    max_iter=ALS_MAX_ITER_VALUE,
    reg_param=ALS_REG_PARAM_VALUE,
)

print('ALS model trained.')
print('User factor rows:', model.userFactors.count())
print('Item factor rows:', model.itemFactors.count())

ALS model trained.
User factor rows: 144
Item factor rows: 47170


## ***Recommendation***

In [41]:
top_n = 10

In [42]:
# Create Spark DataFrame for target user
sample_user_df = spark.createDataFrame(
    [(SAMPLE_USER_ID,)],
    ['user_id']
)

In [43]:
# Generate recommendations directly from live ALS model
user_recommendations = model.recommendForUserSubset(
    sample_user_df,
    top_n
)

In [44]:
# Explode recommendation list into rows
recommended_books = (
    user_recommendations
    .selectExpr("user_id", "explode(recommendations) as rec")
    .select(
        "user_id",
        F.col("rec.item_id").alias("item_id"),
        F.col("rec.rating").alias("predicted_rating")
    )
    .join(
        book_lookup.select("item_id", "title", "author", "isbn"),
        on="item_id",
        how="left"
    )
    .orderBy(F.desc("predicted_rating"))
)

In [45]:
print(f"Top {top_n} recommendations for user {SAMPLE_USER_ID}:")
for index, row in enumerate(recommended_books.collect(), start=1):
    print(f"{index}. {row.title} by {row.author} (ISBN: {row.isbn}) | Predicted Rating: {row.predicted_rating:.2f}")

Top 10 recommendations for user 11676:


1. Odas elementales (Libro amigo) by Pablo Neruda (ISBN: 8402068820) | Predicted Rating: 10.01
2. "Tabakokurenie i mozg (\"Chelovek i okruzhaiushchaia sreda\")" by R. P Kleshcheva (ISBN: 5020054542) | Predicted Rating: 10.01
3. Der Kleine Prinz by Antoine de Saint-Exupery (ISBN: 379200027X) | Predicted Rating: 10.01
4. Die Unertragliche Leichtigkeit des Seins...The Unbearable Lightness of Being by Milan Kundera (ISBN: 3596259924) | Predicted Rating: 10.01
5. KÃ?Â¼sschen, KÃ?Â¼sschen. by Roald Dahl (ISBN: 3499108356) | Predicted Rating: 10.01
6. Das WÃ?Â¼ten der ganzen Welt. by Maarten 't Hart (ISBN: 3492225926) | Predicted Rating: 10.01
7. Shadowrun. Deutschland in den Schatten. by Hans Joachim Alpers (ISBN: 3453863593) | Predicted Rating: 10.01
8. Die schÃ?Â¶ne Schrift. by Rafael Chirbes (ISBN: 3453171489) | Predicted Rating: 10.01
9. 50 Engel fÃ?Â¼r das Jahr. Ein Inspirationsbuch. by Anselm GrÃ?Â¼n (ISBN: 3451049023) | Predicted Rating: 10.01
10. Alles kein Problem in der Liebe. Rosa

## ***Predict User***

In [46]:
def predict_book_rating_kaggle(model, ratings_df, isbn_mapping_df, user_id, isbn):
    normalized_isbn = isbn.strip()

    matched_book = (
        isbn_mapping_df
        .filter(F.col("isbn") == normalized_isbn)
        .select("item_id")
        .limit(1)
        .collect()
    )

    if not matched_book:
        return f"ISBN {normalized_isbn} not found in training data."

    item_id = matched_book[0]["item_id"]

    existing_rating = (
        ratings_df
        .filter(
            (F.col("user_id") == user_id) &
            (F.col("isbn") == normalized_isbn)
        )
        .select("rating")
        .limit(1)
        .collect()
    )

    if existing_rating:
        rating_value = existing_rating[0]["rating"]
        if float(rating_value).is_integer():
            return f"Already Rated. Rating: {int(rating_value)}"
        return f"Already Rated. Rating: {rating_value:.2f}"

    prediction_input = spark.createDataFrame(
        [(user_id, item_id)],
        ["user_id", "item_id"]
    )

    prediction_result = model.transform(prediction_input).collect()

    if not prediction_result:
        return "No Predictions"

    predicted_rating = prediction_result[0]["prediction"]

    if predicted_rating is None or math.isnan(float(predicted_rating)):
        return "No Predictions"

    clipped_prediction = min(
        MAX_RATING,
        max(MIN_RATING, predicted_rating)
    )

    if float(clipped_prediction).is_integer():
        return f"Rating: {int(clipped_prediction)}"

    return f"Rating: {clipped_prediction:.2f}"

In [47]:
prediction = predict_book_rating_kaggle(
    model=model,
    ratings_df=ratings,
    isbn_mapping_df=isbn_mapping,
    user_id=SAMPLE_USER_ID,
    isbn=SAMPLE_ISBN
)

print(f"Prediction for user {SAMPLE_USER_ID}, ISBN {SAMPLE_ISBN}:")
print(prediction)

Prediction for user 11676, ISBN 0001944711:
Already Rated. Rating: 6


# ***Part 3***

## ***Sampling***

In [48]:
def maybe_sample_for_model_comparison(
    ratings_df,
    target_rows=MODEL_COMPARISON_TARGET_ROWS,
    seed=ALS_SEED_VALUE
):
    total_rows = ratings_df.count()

    if total_rows <= target_rows:
        sampled = ratings_df.cache()
        sampled_rows = total_rows
        sample_fraction = 1.0

    else:
        sample_fraction = target_rows / total_rows

        sampled = (
            ratings_df
            .sample(
                withReplacement=False,
                fraction=sample_fraction,
                seed=seed
            )
            .cache()
        )

        sampled_rows = sampled.count()

    return sampled, {
        "total_rows": total_rows,
        "sampled_rows": sampled_rows,
        "sample_fraction": round(sample_fraction, 4),
    }

## ***Book Feature Table***

In [49]:
def build_book_feature_table(indexed_books_df):
    return (
        indexed_books_df
        .select(
            "item_id",

            F.coalesce(
                F.col("author"),
                F.lit("Unknown Author")
            ).alias("author"),

            F.when(
                F.col("year_of_publication")
                .cast("int")
                .between(1800, int(CURRENT_YEAR)),

                F.col("year_of_publication")
                .cast("double"),
            ).alias("publication_year"),
        )

        .dropDuplicates(["item_id"])
        .cache()
    )

## ***Feature Engineering***

In [50]:
def build_feature_frame(
    target_ratings_df,
    stats_source_df,
    book_features_df,
    users_df
):
    # Global defaults
    global_rating = (
        stats_source_df
        .select(F.avg("rating").alias("avg_rating"))
        .first()
        .avg_rating
    )
    global_rating = float(global_rating) if global_rating is not None else 0.0

    average_age = (
        users_df
        .select(
            F.avg(
                F.col("age").cast("double")
            ).alias("avg_age")
        )
        .first()
        .avg_age
    )
    average_age = float(average_age) if average_age is not None else 35.0

    average_year = (
        book_features_df
        .select(
            F.avg("publication_year").alias("avg_year")
        )
        .first()
        .avg_year
    )
    average_year = float(average_year) if average_year is not None else 2000.0

    # User stats
    user_stats = (
        stats_source_df
        .groupBy("user_id")
        .agg(
            F.avg("rating").alias("user_avg_rating"),
            F.count("*").alias("user_rating_count"),
        )
    )

    # Item stats
    item_stats = (
        stats_source_df
        .groupBy("item_id")
        .agg(
            F.avg("rating").alias("item_avg_rating"),
            F.count("*").alias("item_rating_count"),
        )
    )

    # Author stats
    author_stats = (
        stats_source_df
        .join(
            book_features_df.select("item_id", "author"),
            on="item_id",
            how="left"
        )
        .groupBy("author")
        .agg(
            F.avg("rating").alias("author_avg_rating"),
            F.count("*").alias("author_rating_count"),
        )
    )

    # Final feature frame
    feature_frame = (
        target_ratings_df

        .join(
            book_features_df,
            on="item_id",
            how="left"
        )

        .join(
            users_df.select(
                "user_id",
                F.col("age").cast("double").alias("age")
            ),
            on="user_id",
            how="left"
        )

        .join(user_stats, on="user_id", how="left")
        .join(item_stats, on="item_id", how="left")
        .join(author_stats, on="author", how="left")

        # Missing fill
        .withColumn(
            "user_avg_rating",
            F.coalesce(
                F.col("user_avg_rating"),
                F.lit(global_rating)
            )
        )

        .withColumn(
            "item_avg_rating",
            F.coalesce(
                F.col("item_avg_rating"),
                F.lit(global_rating)
            )
        )

        .withColumn(
            "author_avg_rating",
            F.coalesce(
                F.col("author_avg_rating"),
                F.lit(global_rating)
            )
        )

        # Activity logs
        .withColumn(
            "user_rating_count_log",
            F.log1p(
                F.coalesce(
                    F.col("user_rating_count").cast("double"),
                    F.lit(0.0)
                )
            )
        )

        .withColumn(
            "item_rating_count_log",
            F.log1p(
                F.coalesce(
                    F.col("item_rating_count").cast("double"),
                    F.lit(0.0)
                )
            )
        )

        .withColumn(
            "author_rating_count_log",
            F.log1p(
                F.coalesce(
                    F.col("author_rating_count").cast("double"),
                    F.lit(0.0)
                )
            )
        )

        # Missing demographics
        .withColumn(
            "age_filled",
            F.coalesce(
                F.col("age"),
                F.lit(average_age)
            )
        )

        .withColumn(
            "publication_year_filled",
            F.coalesce(
                F.col("publication_year"),
                F.lit(average_year)
            )
        )

        # Derived features
        .withColumn(
            "user_item_mean_gap",
            F.col("user_avg_rating") - F.col("item_avg_rating")
        )

        .withColumn(
            "book_age",
            F.lit(CURRENT_YEAR) - F.col("publication_year_filled")
        )
    )

    return (
        feature_frame
        .select(
            "user_id",
            "item_id",
            "rating",
            *MODEL_FEATURE_COLUMNS
        )
        .cache()
    )

## ***FMRegressor***

In [51]:
def build_fm_pipeline(
    factor_size,
    reg_param,
    step_size
):
    assembler = VectorAssembler(
        inputCols=MODEL_FEATURE_COLUMNS,
        outputCol="raw_features"
    )

    scaler = StandardScaler(
        inputCol="raw_features",
        outputCol="features",
        withStd=True,
        withMean=True
    )

    estimator = FMRegressor(
        featuresCol="features",
        labelCol="rating",
        predictionCol="prediction",
        factorSize=factor_size,
        regParam=reg_param,
        stepSize=step_size,
        miniBatchFraction=1.0,
        solver="adamW",
        maxIter=80,
        seed=ALS_SEED_VALUE,
    )

    return Pipeline(
        stages=[
            assembler,
            scaler,
            estimator
        ]
    )

## ***XGB Data Preparation***

In [52]:
def collect_ranker_dataset(feature_frame):
    ordered_rows = (
        feature_frame
        .orderBy("user_id", "item_id")
        .collect()
    )

    feature_rows = []
    feature_matrix = []
    labels = []
    group_sizes = []

    current_user = None
    current_group_size = 0

    for row in ordered_rows:
        user_id = int(row.user_id)

        if current_user is None:
            current_user = user_id

        elif user_id != current_user:
            group_sizes.append(current_group_size)
            current_group_size = 0
            current_user = user_id

        current_group_size += 1

        feature_rows.append({
            "user_id": user_id,
            "item_id": int(row.item_id),
            "rating": float(row.rating),
        })

        feature_matrix.append([
            float(getattr(row, feature_name))
            for feature_name in MODEL_FEATURE_COLUMNS
        ])

        labels.append(float(row.rating))

    if current_group_size:
        group_sizes.append(current_group_size)

    return (
        feature_rows,
        feature_matrix,
        labels,
        group_sizes
    )

## ***XGB Trainer***

In [53]:
def fit_xgb_ranker_model(
    train_frame,
    n_estimators,
    max_depth,
    learning_rate,
    min_child_weight
):
    train_rows, train_features, train_labels, train_group_sizes = (
        collect_ranker_dataset(train_frame)
    )

    if not train_rows:
        raise ValueError("XGBRanker training frame empty.")

    ranker = XGBRanker(
        objective="rank:ndcg",
        eval_metric=f"ndcg@{RANKING_TOP_K}",
        tree_method="hist",
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        min_child_weight=min_child_weight,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=ALS_SEED_VALUE,
        n_jobs=1,
    )

    ranker.fit(
        train_features,
        train_labels,
        group=train_group_sizes
    )

    return ranker

## ***Ranking Helper***

In [54]:
def rating_gain(
    value,
    threshold=RELEVANT_RATING_THRESHOLD
):
    return max(
        float(value) - float(threshold) + 1.0,
        0.0
    )


def ranking_summary_key(row):
    return (
        row["ndcg_at_k"] if row["ndcg_at_k"] is not None else -1.0,
        row["map_at_k"] if row["map_at_k"] is not None else -1.0,
        row["precision_at_k"] if row["precision_at_k"] is not None else -1.0,
        row["recall_at_k"] if row["recall_at_k"] is not None else -1.0,
        row["coverage"],
    )

In [55]:
def ranking_summary_key(row):
    return (
        row["ndcg_at_k"] if row["ndcg_at_k"] is not None else -1.0,
        row["map_at_k"] if row["map_at_k"] is not None else -1.0,
        row["precision_at_k"] if row["precision_at_k"] is not None else -1.0,
        row["recall_at_k"] if row["recall_at_k"] is not None else -1.0,
        row["coverage"],
    )

# ***Core Ranking Scorer***

In [56]:
def score_ranking_rows(
    scored_rows,
    expected_rows,
    top_k=RANKING_TOP_K
):
    coverage = (
        round(len(scored_rows) / expected_rows, 4)
        if expected_rows else 0.0
    )

    grouped_rows = {}

    for row in scored_rows:
        grouped_rows.setdefault(
            int(row["user_id"]),
            []
        ).append({
            "item_id": int(row["item_id"]),
            "rating": float(row["rating"]),
            "prediction": float(row["prediction"]),
        })

    ndcg_values = []
    map_values = []
    precision_values = []
    recall_values = []

    eligible_users = 0

    for user_rows in grouped_rows.values():

        relevant_items = {
            row["item_id"]
            for row in user_rows
            if row["rating"] >= RELEVANT_RATING_THRESHOLD
        }

        if not relevant_items:
            continue

        eligible_users += 1

        ranked_rows = sorted(
            user_rows,
            key=lambda row: (
                -row["prediction"],
                -row["rating"],
                row["item_id"]
            ),
        )

        top_rows = ranked_rows[:top_k]
        effective_k = len(top_rows)

        hits = 0
        precision_sum = 0.0
        dcg = 0.0

        for index, row in enumerate(top_rows, start=1):

            if row["item_id"] in relevant_items:
                hits += 1
                precision_sum += hits / index

            gain = rating_gain(row["rating"])

            if gain > 0:
                dcg += gain / math.log2(index + 1)

        ideal_gains = sorted(
            (
                rating_gain(row["rating"])
                for row in user_rows
            ),
            reverse=True
        )

        ideal_gains = [
            gain
            for gain in ideal_gains
            if gain > 0
        ][:top_k]

        idcg = sum(
            gain / math.log2(index + 1)
            for index, gain in enumerate(
                ideal_gains,
                start=1
            )
        )

        precision_values.append(
            hits / effective_k if effective_k else 0.0
        )

        recall_values.append(
            hits / len(relevant_items)
        )

        map_denominator = min(
            len(relevant_items),
            top_k
        )

        map_values.append(
            precision_sum / map_denominator
            if map_denominator else 0.0
        )

        ndcg_values.append(
            dcg / idcg if idcg else 0.0
        )

    return {
        "scored_rows": len(scored_rows),
        "coverage": coverage,
        "users_scored": len(grouped_rows),
        "eligible_users": eligible_users,
        "ndcg_at_k": round(
            sum(ndcg_values) / len(ndcg_values),
            4
        ) if ndcg_values else None,
        "map_at_k": round(
            sum(map_values) / len(map_values),
            4
        ) if map_values else None,
        "precision_at_k": round(
            sum(precision_values) / len(precision_values),
            4
        ) if precision_values else None,
        "recall_at_k": round(
            sum(recall_values) / len(recall_values),
            4
        ) if recall_values else None,
    }

## ***ALS + FMR Scorer***

In [57]:
def score_ranking_predictions(
    predictions_df,
    expected_rows,
    top_k=RANKING_TOP_K
):
    scored_rows = [
        {
            "user_id": int(row.user_id),
            "item_id": int(row.item_id),
            "rating": float(row.rating),
            "prediction": float(row.prediction),
        }

        for row in (
            predictions_df
            .filter(
                F.col("prediction").isNotNull() &
                (~F.isnan("prediction"))
            )
            .select(
                "user_id",
                "item_id",
                "rating",
                "prediction"
            )
            .collect()
        )
    ]

    return score_ranking_rows(
        scored_rows,
        expected_rows,
        top_k=top_k
    )

## ***XGB Scorer***

In [58]:
def score_local_xgb_ranker(
    model,
    feature_frame
):
    feature_rows, feature_matrix, _, _ = (
        collect_ranker_dataset(feature_frame)
    )

    predictions = (
        model.predict(feature_matrix)
        if feature_matrix else []
    )

    scored_rows = [
        {
            **feature_row,
            "prediction": float(prediction),
        }
        for feature_row, prediction in zip(
            feature_rows,
            predictions
        )
    ]

    return score_ranking_rows(
        scored_rows,
        expected_rows=len(feature_rows)
    )

## ***FM + ALS Tuner***

In [59]:
def tune_pipeline_model(
    model_name,
    param_grid,
    pipeline_builder,
    train_frame,
    validation_frame
):
    validation_rows = validation_frame.count()

    results = []
    best_result = None

    for params in param_grid:

        candidate_model = (
            pipeline_builder(**params)
            .fit(train_frame)
        )

        metrics = score_ranking_predictions(
            candidate_model.transform(validation_frame),
            validation_rows
        )

        row = {
            "model_name": model_name,
            "params": params,
            "params_json": json.dumps(
                params,
                sort_keys=True
            ),
            **metrics,
        }

        results.append(row)

        if row["ndcg_at_k"] is None:
            continue

        if (
            best_result is None or
            ranking_summary_key(row) >
            ranking_summary_key(best_result)
        ):
            best_result = row

    if best_result is None:
        raise ValueError(
            f"No valid validation predictions for {model_name}."
        )

    return best_result, results

## ***Evaluation***

In [60]:
def evaluate_pipeline_model(
    model_name,
    best_params,
    pipeline_builder,
    train_frame,
    test_frame
):
    fitted_model = (
        pipeline_builder(**best_params)
        .fit(train_frame)
    )

    test_rows = test_frame.count()

    metrics = score_ranking_predictions(
        fitted_model.transform(test_frame),
        test_rows
    )

    return {
        "model_name": model_name,
        "best_params": best_params,
        "best_params_json": json.dumps(
            best_params,
            sort_keys=True
        ),
        **metrics,
    }

## ***XGB Validation Tuner***

In [61]:
def tune_xgb_ranker_model(
    train_frame,
    validation_frame
):
    results = []
    best_result = None

    for params in XGB_RANKER_GRID:

        candidate_model = fit_xgb_ranker_model(
            train_frame,
            **params
        )

        metrics = score_local_xgb_ranker(
            candidate_model,
            validation_frame
        )

        row = {
            "model_name": "XGBRanker",
            "params": params,
            "params_json": json.dumps(
                params,
                sort_keys=True
            ),
            **metrics,
        }

        results.append(row)

        if row["ndcg_at_k"] is None:
            continue

        if (
            best_result is None or
            ranking_summary_key(row) >
            ranking_summary_key(best_result)
        ):
            best_result = row

    if best_result is None:
        raise ValueError(
            "No valid validation predictions for XGBRanker."
        )

    return best_result, results

## ***XGB Final Test Evaluation***

In [62]:
def evaluate_xgb_ranker_model(
    best_params,
    train_frame,
    test_frame
):
    fitted_model = fit_xgb_ranker_model(
        train_frame,
        **best_params
    )

    metrics = score_local_xgb_ranker(
        fitted_model,
        test_frame
    )

    return {
        "model_name": "XGBRanker",
        "best_params": best_params,
        "best_params_json": json.dumps(
            best_params,
            sort_keys=True
        ),
        **metrics,
    }

## ***Results***

In [63]:
def print_metric_rows(
    title,
    rows,
    params_key
):
    print(f"\n{title}")

    for index, row in enumerate(rows, start=1):
        print(
            f"{index}. {row['model_name']} | "
            f"params={row[params_key]} | "
            f"NDCG@{RANKING_TOP_K}={row['ndcg_at_k']} | "
            f"MAP@{RANKING_TOP_K}={row['map_at_k']} | "
            f"Precision@{RANKING_TOP_K}={row['precision_at_k']} | "
            f"Recall@{RANKING_TOP_K}={row['recall_at_k']} | "
            f"coverage={row['coverage']} | "
            f"eligible_users={row['eligible_users']} | "
            f"scored_rows={row['scored_rows']}"
        )

# ***Part 4***

## ***Sample***

In [64]:
modeling_ratings, modeling_summary = maybe_sample_for_model_comparison(
    indexed_ratings,
    target_rows=MODEL_COMPARISON_TARGET_ROWS,
    seed=ALS_SEED_VALUE,
)

In [65]:
print("Comparison Dataset Summary:")
print(json.dumps(modeling_summary, indent=2))

Comparison Dataset Summary:
{
  "total_rows": 62191,
  "sampled_rows": 25075,
  "sample_fraction": 0.402
}


## ***Train / Test / Val Split***

In [66]:
tune_train_ratings, validation_ratings, test_ratings = modeling_ratings.randomSplit(
    MODEL_COMPARISON_SPLIT,
    seed=ALS_SEED_VALUE
)

tune_train_ratings = tune_train_ratings.cache()
validation_ratings = validation_ratings.cache()
test_ratings = test_ratings.cache()

split_summary = {
    **modeling_summary,
    "ranking_top_k": RANKING_TOP_K,
    "relevant_rating_threshold": RELEVANT_RATING_THRESHOLD,
    "tune_train_rows": tune_train_ratings.count(),
    "validation_rows": validation_ratings.count(),
    "test_rows": test_ratings.count(),
}

print("Split Summary:")
print(json.dumps(split_summary, indent=2))

if min(
    split_summary["tune_train_rows"],
    split_summary["validation_rows"],
    split_summary["test_rows"]
) == 0:
    raise ValueError(
        "Model comparison split produced empty partition. "
        "Increase sample size or adjust split."
    )

Split Summary:
{
  "total_rows": 62191,
  "sampled_rows": 25075,
  "sample_fraction": 0.402,
  "ranking_top_k": 10,
  "relevant_rating_threshold": 8.0,
  "tune_train_rows": 17605,
  "validation_rows": 3725,
  "test_rows": 3745
}


## ***ALS Eval***

In [67]:
tuning_results = []

In [68]:
als_best = None
validation_count = split_summary["validation_rows"]

In [69]:
for params in ALS_TUNING_GRID:

    candidate_model = train_als_model(
        tune_train_ratings,
        rank=params["rank"],
        max_iter=params["max_iter"],
        reg_param=params["reg_param"],
    )

    metrics = score_ranking_predictions(
        candidate_model.transform(validation_ratings),
        validation_count
    )

    row = {
        "model_name": "ALS",
        "params": params,
        "params_json": json.dumps(
            params,
            sort_keys=True
        ),
        **metrics,
    }

    tuning_results.append(row)

    print(
        f"ALS params={row['params_json']} -> "
        f"NDCG@{RANKING_TOP_K}={row['ndcg_at_k']} | "
        f"MAP@{RANKING_TOP_K}={row['map_at_k']} | "
        f"Precision@{RANKING_TOP_K}={row['precision_at_k']} | "
        f"Coverage={row['coverage']}"
    )

    if row["ndcg_at_k"] is None:
        continue

    if (
        als_best is None or
        ranking_summary_key(row) >
        ranking_summary_key(als_best)
    ):
        als_best = row

if als_best is None:
    raise ValueError(
        "ALS tuning produced no valid validation predictions."
    )

ALS params={"max_iter": 10, "rank": 12, "reg_param": 0.05} -> NDCG@10=0.8428 | MAP@10=0.832 | Precision@10=0.733 | Coverage=0.1844
ALS params={"max_iter": 10, "rank": 12, "reg_param": 0.1} -> NDCG@10=0.8325 | MAP@10=0.8263 | Precision@10=0.7321 | Coverage=0.1844
ALS params={"max_iter": 15, "rank": 12, "reg_param": 0.05} -> NDCG@10=0.8472 | MAP@10=0.8356 | Precision@10=0.734 | Coverage=0.1844
ALS params={"max_iter": 15, "rank": 12, "reg_param": 0.1} -> NDCG@10=0.8424 | MAP@10=0.8352 | Precision@10=0.7321 | Coverage=0.1844
ALS params={"max_iter": 10, "rank": 24, "reg_param": 0.05} -> NDCG@10=0.8458 | MAP@10=0.8323 | Precision@10=0.7321 | Coverage=0.1844
ALS params={"max_iter": 10, "rank": 24, "reg_param": 0.1} -> NDCG@10=0.838 | MAP@10=0.8211 | Precision@10=0.7321 | Coverage=0.1844
ALS params={"max_iter": 15, "rank": 24, "reg_param": 0.05} -> NDCG@10=0.8439 | MAP@10=0.8307 | Precision@10=0.733 | Coverage=0.1844
ALS params={"max_iter": 15, "rank": 24, "reg_param": 0.1} -> NDCG@10=0.8419 |

## ***Feature Engineering for FM and XGB***

In [70]:
book_features = build_book_feature_table(indexed_books)

In [71]:
feature_train_frame = build_feature_frame(
    tune_train_ratings,
    tune_train_ratings,
    book_features,
    users
)

In [72]:
feature_validation_frame = build_feature_frame(
    validation_ratings,
    tune_train_ratings,
    book_features,
    users
)

## ***FMR Eval***

In [73]:
fm_best, fm_tuning_rows = tune_pipeline_model(
    model_name="FMRegressor",
    param_grid=FM_GRID,
    pipeline_builder=build_fm_pipeline,
    train_frame=feature_train_frame,
    validation_frame=feature_validation_frame,
)

tuning_results.extend(fm_tuning_rows)

26/05/02 07:49:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## ***XGBR Eval***

In [74]:
xgb_best, xgb_tuning_rows = tune_xgb_ranker_model(
    feature_train_frame,
    feature_validation_frame,
)

tuning_results.extend(xgb_tuning_rows)

## ***Final Training***

In [75]:
final_train_ratings = (
    tune_train_ratings
    .unionByName(validation_ratings)
    .cache()
)

## ***ALS Test***

In [76]:
final_results = []

als_final_model = train_als_model(
    final_train_ratings,
    rank=als_best["params"]["rank"],
    max_iter=als_best["params"]["max_iter"],
    reg_param=als_best["params"]["reg_param"],
)

als_final_metrics = score_ranking_predictions(
    als_final_model.transform(test_ratings),
    split_summary["test_rows"]
)

final_results.append({
    "model_name": "ALS",
    "best_params": als_best["params"],
    "best_params_json": als_best["params_json"],
    **als_final_metrics,
})

## ***Final Feature Table***

In [77]:
final_train_features = build_feature_frame(
    final_train_ratings,
    final_train_ratings,
    book_features,
    users
)

test_features = build_feature_frame(
    test_ratings,
    final_train_ratings,
    book_features,
    users
)

## ***Final FM Test***

In [78]:
final_results.append(
    evaluate_pipeline_model(
        model_name="FMRegressor",
        best_params=fm_best["params"],
        pipeline_builder=build_fm_pipeline,
        train_frame=final_train_features,
        test_frame=test_features,
    )
)

## ***Final XGB Test***

In [79]:
final_results.append(
    evaluate_xgb_ranker_model(
        best_params=xgb_best["params"],
        train_frame=final_train_features,
        test_frame=test_features,
    )
)

## ***Validation Table***

In [80]:
tuning_table = [
    {
        "model_name": row["model_name"],
        "params_json": row["params_json"],
        "ndcg_at_k": row["ndcg_at_k"],
        "map_at_k": row["map_at_k"],
        "precision_at_k": row["precision_at_k"],
        "recall_at_k": row["recall_at_k"],
        "coverage": row["coverage"],
        "eligible_users": row["eligible_users"],
        "users_scored": row["users_scored"],
        "scored_rows": row["scored_rows"],
    }
    for row in sorted(
        tuning_results,
        key=ranking_summary_key,
        reverse=True
    )
]

## ***Final Test***

In [81]:
final_table = [
    {
        "model_name": row["model_name"],
        "best_params_json": row["best_params_json"],
        "ndcg_at_k": row["ndcg_at_k"],
        "map_at_k": row["map_at_k"],
        "precision_at_k": row["precision_at_k"],
        "recall_at_k": row["recall_at_k"],
        "coverage": row["coverage"],
        "eligible_users": row["eligible_users"],
        "users_scored": row["users_scored"],
        "scored_rows": row["scored_rows"],
    }
    for row in sorted(
        final_results,
        key=ranking_summary_key,
        reverse=True
    )
]

## ***Results***

In [82]:
print_metric_rows(
    "VALIDATION TUNING RESULTS",
    tuning_table,
    "params_json"
)

print_metric_rows(
    "FINAL TEST RESULTS",
    final_table,
    "best_params_json"
)


VALIDATION TUNING RESULTS
1. ALS | params={"max_iter": 15, "rank": 12, "reg_param": 0.05} | NDCG@10=0.8472 | MAP@10=0.8356 | Precision@10=0.734 | Recall@10=0.9693 | coverage=0.1844 | eligible_users=106 | scored_rows=687
2. ALS | params={"max_iter": 10, "rank": 24, "reg_param": 0.05} | NDCG@10=0.8458 | MAP@10=0.8323 | Precision@10=0.7321 | Recall@10=0.9666 | coverage=0.1844 | eligible_users=106 | scored_rows=687
3. ALS | params={"max_iter": 15, "rank": 24, "reg_param": 0.05} | NDCG@10=0.8439 | MAP@10=0.8307 | Precision@10=0.733 | Recall@10=0.967 | coverage=0.1844 | eligible_users=106 | scored_rows=687
4. ALS | params={"max_iter": 10, "rank": 12, "reg_param": 0.05} | NDCG@10=0.8428 | MAP@10=0.832 | Precision@10=0.733 | Recall@10=0.9689 | coverage=0.1844 | eligible_users=106 | scored_rows=687
5. ALS | params={"max_iter": 15, "rank": 12, "reg_param": 0.1} | NDCG@10=0.8424 | MAP@10=0.8352 | Precision@10=0.7321 | Recall@10=0.9677 | coverage=0.1844 | eligible_users=106 | scored_rows=687
6. A

## ***Summary***

In [83]:
model_comparison_summary = {
    "split_summary": split_summary,
    "best_validation_configs": {
        "ALS": als_best["params"],
        "FMRegressor": fm_best["params"],
        "XGBRanker": xgb_best["params"],
    },
    "final_test_results": final_table,
}

In [84]:
print("\nMODEL COMPARISON SUMMARY:")
print(json.dumps(model_comparison_summary, indent=2))


MODEL COMPARISON SUMMARY:
{
  "split_summary": {
    "total_rows": 62191,
    "sampled_rows": 25075,
    "sample_fraction": 0.402,
    "ranking_top_k": 10,
    "relevant_rating_threshold": 8.0,
    "tune_train_rows": 17605,
    "validation_rows": 3725,
    "test_rows": 3745
  },
  "best_validation_configs": {
    "ALS": {
      "rank": 12,
      "max_iter": 15,
      "reg_param": 0.05
    },
    "FMRegressor": {
      "factor_size": 8,
      "reg_param": 0.0,
      "step_size": 0.05
    },
    "XGBRanker": {
      "n_estimators": 30,
      "max_depth": 4,
      "learning_rate": 0.05,
      "min_child_weight": 1
    }
  },
  "final_test_results": [
    {
      "model_name": "ALS",
      "best_params_json": "{\"max_iter\": 15, \"rank\": 12, \"reg_param\": 0.05}",
      "ndcg_at_k": 0.8416,
      "map_at_k": 0.8152,
      "precision_at_k": 0.7194,
      "recall_at_k": 0.9598,
      "coverage": 0.2096,
      "eligible_users": 120,
      "users_scored": 130,
      "scored_rows": 785
    },